In [ ]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.layers import (Input, Embedding
, SpatialDropout1D, LSTM, 
                                     Dense, Dropout, BatchNormalization)
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

from sklearn.utils import resample
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (confusion_matrix, classification_report, 
                             accuracy_score, precision_recall_fscore_support, 
                             roc_curve, auc)
import gensim.downloader as api

# Downloads
nltk.download('stopwords')
nltk.download('wordnet')

In [ ]:
# Load Dataset
df = pd.read_csv(r"C:\MH_Sentiment_Project\Combined Data.csv")
if 'Unnamed: 0' in df.columns:
    df.drop('Unnamed: 0', axis=1, inplace=True)
df.dropna(inplace=True)

# Resampling to Balance all 7 Classes
def resample_data(df):
    max_count = df['status'].value_counts().max()
    df_resampled = pd.DataFrame()
    for status in df['status'].unique():
        df_class = df[df['status'] == status]
        df_class_resampled = resample(df_class, replace=True, n_samples=max_count, random_state=42)
        df_resampled = pd.concat([df_resampled, df_class_resampled])
    return df_resampled

df = resample_data(df)
print("Balanced Classes:\n", df['status'].value_counts())

In [ ]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    text = re.sub(r'[^a-zA-Z0-9\s]', '', str(text).lower())
    tokens = text.split()
    tokens = [lemmatizer.lemmatize(word) for word in tokens if word not in stop_words]
    return " ".join(tokens)

df['statement'] = df['statement'].apply(preprocess_text)

# Label Encoding
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(df['status'].values)
y_categorical = to_categorical(y_encoded)

# Split (90-10)
X_train, X_test, y_train, y_test = train_test_split(df['statement'].values, y_categorical, test_size=0.1, random_state=42)

# Tokenization
max_words = 50000
max_len = 100
tokenizer = Tokenizer(num_words=max_words, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

X_train_padded = pad_sequences(tokenizer.texts_to_sequences(X_train), maxlen=max_len, padding='post')
X_test_padded = pad_sequences(tokenizer.texts_to_sequences(X_test), maxlen=max_len, padding='post')

In [ ]:
print("Loading Word2Vec model...")
word2vec_model = api.load("word2vec-google-news-300")
embedding_dim = 300
num_words = min(max_words, len(tokenizer.word_index) + 1)
embedding_matrix = np.zeros((num_words, embedding_dim))

for word, i in tokenizer.word_index.items():
    if i < max_words:
        if word in word2vec_model:
            embedding_matrix[i] = word2vec_model[word]
        else:
            embedding_matrix[i] = np.random.normal(scale=0.6, size=(embedding_dim,))
print("Embedding Matrix ready.")

In [ ]:
def build_lstm():
    model = Sequential([
        # Embedding Layer
        Embedding(num_words, embedding_dim, weights=[embedding_matrix], 
                  input_length=max_len, trainable=False),
        
        # Spatial Dropout for better NLP regularization
        SpatialDropout1D(0.4),
        
        # LSTM Layer 1 (returns sequences for the next LSTM layer)
        LSTM(128, return_sequences=True, dropout=0.2, recurrent_dropout=0.2),
        
        # LSTM Layer 2 (final sequence processing)
        LSTM(64, dropout=0.2),
        
        # Fully Connected Layers
        Dense(64, activation='relu'),
        BatchNormalization(),
        Dropout(0.3),
        
        # Output Layer
        Dense(7, activation='softmax')
    ], name="Deep_LSTM_Model")
    return model

model = build_lstm()
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

In [ ]:
callbacks = [
    EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2)
]

# Training the LSTM model
history = model.fit(
    X_train_padded, y_train,
    validation_data=(X_test_padded, y_test),
    epochs=12,
    batch_size=128,
    callbacks=callbacks
)

# Saving the model in both formats
model.save("sentiment analysis lstm.keras")
model.save("mental_health_model lstm.h5")
print("LSTM Model saved successfully.")

In [ ]:
y_pred_probs = model.predict(X_test_padded)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = np.argmax(y_test, axis=1)

# Metrics Calculation
precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average=None)
performance_df = pd.DataFrame({
    'Emotion': label_encoder.classes_,
    'Precision': precision, 'Recall': recall, 'F1-Score': f1
})

print("\n--- LSTM PERFORMANCE TABLE ---")
print(performance_df)
print("\n--- CLASSIFICATION REPORT ---")
print(classification_report(y_true, y_pred, target_names=label_encoder.classes_))

In [ ]:
plt.figure(figsize=(16, 5))

# Plot Accuracy & Loss
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Val Accuracy')
plt.title('LSTM Accuracy Curves')
plt.legend()

# Plot Confusion Matrix
plt.subplot(1, 2, 2)
cm = confusion_matrix(y_true, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Purples', 
            xticklabels=label_encoder.classes_, yticklabels=label_encoder.classes_)
plt.title('LSTM Confusion Matrix Heatmap')
plt.show()

In [ ]:
plt.figure(figsize=(10, 8))
for i in range(7):
    fpr, tpr, _ = roc_curve(y_test[:, i], y_pred_probs[:, i])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f'{label_encoder.classes_[i]} (AUC = {roc_auc:.2f})')

plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves (LSTM)')
plt.legend(loc="lower right")
plt.show()

In [ ]:
def analyze_mental_health(text):
    """
    Function to preprocess text, predict mental health status using LSTM,
    and calculate intensity levels based on confidence.
    """
    clean_text = preprocess_text(text)
    seq = tokenizer.texts_to_sequences([clean_text])
    padded = pad_sequences(seq, maxlen=max_len, padding='post')
    
    prediction = model.predict(padded, verbose=0)[0]
    class_idx = np.argmax(prediction)
    emotion = label_encoder.classes_[class_idx]
    confidence = prediction[class_idx]
    
    # Intensity Logic
    if confidence > 0.80:
        intensity = "High / Acute"
    elif confidence > 0.50:
        intensity = "Moderate"
    else:
        intensity = "Low / Emerging"
        
    print(f"\n--- Mental Health Analysis (LSTM) ---")
    print(f"Post: {text}")
    print(f"Detected: {emotion}")
    print(f"Intensity: {intensity} ({confidence*100:.1f}%)")
    print("-" * 40)

# --- DIRECT INPUT TEST CASES ---
print("[RUNNING DIRECT TEST CASES FOR LSTM]")

# Static Test Cases
analyze_mental_health("I can't stop my hands from shaking and I feel like something terrible is about to happen.")
analyze_mental_health("I have so much work to do and I feel completely overwhelmed by the deadlines.")
analyze_mental_health("I feel like I'm drowning and no one can hear me. I don't want to do this anymore.")

print("\nLSTM Model Analysis Complete.")